In [18]:
import gc
import json
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn

from IPython.display import display

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import f_classif
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold

print("scikit-learn version:", sklearn.__version__)

scikit-learn version: 1.9.0


In [19]:
DATASET_NAME = "openSMILE 0.5s"

DATASET_PATH = Path(
    "/Users/bhavaykhatri/Desktop/embeddings/openSMILE/"
    "singBAP_dataset_opensmile-compare-2016_0.5s.parquet"
)

OUTPUT_DIR = Path(
    "opensmile_0_5s_optimized_feature_selection"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TARGET_EXPERIENCE = [
    "intermediate",
    "professional",
]

TARGET_CLASSES = [
    "correct",
    "arched_back",
    "hunched_back",
    "sideways",
    "chest_breathing",
    "over_articulation",
    "under_articulation",
]

RANDOM_STATE = 42
INNER_RANDOM_STATE = 43

ORIGINAL_FEATURE_COUNT = 6373

BASELINE_ACCURACY = 0.3291
BASELINE_BALANCED_ACCURACY = 0.3437
BASELINE_MACRO_F1 = 0.3242

# Feature counts worth testing based on your earlier results
ANOVA_K_VALUES = [
    250,
    500,
    750,
    1000,
    1250,
    1500,
]

# Faster forest used only for feature-count screening
SEARCH_RF_TREES = 150

SEARCH_RESULTS_PATH = (
    OUTPUT_DIR
    / "anova_search_checkpoint.csv"
)

TUNING_RESULTS_PATH = (
    OUTPUT_DIR
    / "rf_tuning_checkpoint.csv"
)

print("Dataset:", DATASET_NAME)
print("Dataset path:", DATASET_PATH)
print("Output directory:", OUTPUT_DIR.resolve())

Dataset: openSMILE 0.5s
Dataset path: /Users/bhavaykhatri/Desktop/embeddings/openSMILE/singBAP_dataset_opensmile-compare-2016_0.5s.parquet
Output directory: /Users/bhavaykhatri/Desktop/Assignments/audio_data_benchmarking_mml_lab/singBAP_Baselines-and-feature selection/opensmile0.5s/opensmile_0_5s_optimized_feature_selection


In [20]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found:\n{DATASET_PATH}"
    )

df = pd.read_parquet(DATASET_PATH)

required_columns = {
    "condition",
    "experience",
    "filename",
    "embedding",
}

missing_columns = (
    required_columns
    - set(df.columns)
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: "
        f"{sorted(missing_columns)}"
    )

print("Original shape:", df.shape)

df = df[
    df["experience"].isin(
        TARGET_EXPERIENCE
    )
].copy()

df = df[
    df["condition"].isin(
        TARGET_CLASSES
    )
].copy()

df = df.reset_index(drop=True)

print("Filtered shape:", df.shape)

print("\nClass distribution:")

display(
    df["condition"]
    .value_counts()
    .reindex(TARGET_CLASSES)
    .rename_axis("Class")
    .to_frame("Samples")
)

print("\nExperience distribution:")

display(
    df["experience"]
    .value_counts()
    .rename_axis("Experience")
    .to_frame("Samples")
)

Original shape: (34409, 13)
Filtered shape: (28418, 13)

Class distribution:


,Samples
Class,
correct,5137
arched_back,3362
hunched_back,4540
sideways,4285
chest_breathing,4259
over_articulation,3454
under_articulation,3381



Experience distribution:


,Samples
Experience,
intermediate,23420
professional,4998


In [21]:
def decode_embedding(value):
    if isinstance(
        value,
        (bytes, bytearray, memoryview),
    ):
        return np.frombuffer(
            value,
            dtype=np.float32,
        ).copy()

    return np.asarray(
        value,
        dtype=np.float32,
    ).reshape(-1)


X = np.ascontiguousarray(
    np.vstack(
        df["embedding"].map(
            decode_embedding
        )
    ),
    dtype=np.float32,
)

y = (
    df["condition"]
    .astype(str)
    .to_numpy()
)

groups = (
    df["filename"]
    .astype(str)
    .to_numpy()
)

if X.ndim != 2:
    raise ValueError(
        f"Expected a 2D feature matrix, "
        f"received shape {X.shape}"
    )

finite_rows = np.isfinite(X).all(axis=1)

removed_rows = int(
    len(X) - finite_rows.sum()
)

print(
    "Rows containing NaN or infinity:",
    removed_rows,
)

if removed_rows > 0:
    X = X[finite_rows]
    y = y[finite_rows]
    groups = groups[finite_rows]

print("X shape:", X.shape)
print("y shape:", y.shape)
print(
    "Unique recordings:",
    len(np.unique(groups)),
)

if X.shape[1] != ORIGINAL_FEATURE_COUNT:
    print(
        "Warning: expected",
        ORIGINAL_FEATURE_COUNT,
        "features but found",
        X.shape[1],
    )

# Release the original dataframe
del df
gc.collect()

Rows containing NaN or infinity: 0
X shape: (28418, 6373)
y shape: (28418,)
Unique recordings: 3046


33

In [22]:
outer_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

outer_train_idx, test_idx = next(
    outer_splitter.split(
        X,
        y,
        groups=groups,
    )
)

X_outer_train = X[outer_train_idx]
y_outer_train = y[outer_train_idx]
outer_train_groups = groups[
    outer_train_idx
]

X_test = X[test_idx]
y_test = y[test_idx]
test_groups = groups[test_idx]

outer_overlap = (
    set(outer_train_groups)
    & set(test_groups)
)

inner_splitter = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=INNER_RANDOM_STATE,
)

fs_train_relative_idx, validation_relative_idx = next(
    inner_splitter.split(
        X_outer_train,
        y_outer_train,
        groups=outer_train_groups,
    )
)

X_fs_train = X_outer_train[
    fs_train_relative_idx
]

y_fs_train = y_outer_train[
    fs_train_relative_idx
]

fs_train_groups = outer_train_groups[
    fs_train_relative_idx
]

X_validation = X_outer_train[
    validation_relative_idx
]

y_validation = y_outer_train[
    validation_relative_idx
]

validation_groups = outer_train_groups[
    validation_relative_idx
]

inner_overlap = (
    set(fs_train_groups)
    & set(validation_groups)
)

print("Complete dataset:", X.shape)
print("Outer training:", X_outer_train.shape)
print("Final test:", X_test.shape)

print()

print(
    "Feature-selection training:",
    X_fs_train.shape,
)

print(
    "Validation:",
    X_validation.shape,
)

print()

print(
    "Shared outer recordings:",
    len(outer_overlap),
)

print(
    "Shared inner recordings:",
    len(inner_overlap),
)

assert len(outer_overlap) == 0
assert len(inner_overlap) == 0

# X is no longer needed because the outer arrays exist
del X, y, groups
gc.collect()

Complete dataset: (28418, 6373)
Outer training: (22735, 6373)
Final test: (5683, 6373)

Feature-selection training: (17049, 6373)
Validation: (5686, 6373)

Shared outer recordings: 0
Shared inner recordings: 0


0

In [23]:
feature_variances = np.var(
    X_fs_train,
    axis=0,
)

nonconstant_indices = np.flatnonzero(
    feature_variances > 0
).astype(np.int32)

constant_feature_count = (
    X_fs_train.shape[1]
    - len(nonconstant_indices)
)

print(
    "Original features:",
    X_fs_train.shape[1],
)

print(
    "Nonconstant features:",
    len(nonconstant_indices),
)

print(
    "Constant features removed:",
    constant_feature_count,
)

anova_start = time.time()

if (
    len(nonconstant_indices)
    == X_fs_train.shape[1]
):
    X_anova_train = X_fs_train
else:
    X_anova_train = X_fs_train[
        :,
        nonconstant_indices,
    ]

anova_scores, anova_p_values = f_classif(
    X_anova_train,
    y_fs_train,
)

anova_scores = np.nan_to_num(
    anova_scores,
    nan=-np.inf,
    posinf=np.finfo(
        np.float32
    ).max,
    neginf=-np.inf,
)

anova_p_values = np.nan_to_num(
    anova_p_values,
    nan=1.0,
    posinf=1.0,
    neginf=0.0,
)

anova_local_ranking = np.argsort(
    anova_scores
)[::-1]

anova_ranked_indices = (
    nonconstant_indices[
        anova_local_ranking
    ]
)

print(
    "ANOVA ranking time:",
    f"{time.time() - anova_start:.2f}s",
)

print(
    "Ranked features:",
    len(anova_ranked_indices),
)

if X_anova_train is not X_fs_train:
    del X_anova_train

gc.collect()

Original features: 6373
Nonconstant features: 6373
Constant features removed: 0
ANOVA ranking time: 0.19s
Ranked features: 6373


0

In [24]:
def select_columns(
    matrix,
    feature_indices,
):
    feature_indices = np.asarray(
        feature_indices,
        dtype=np.int32,
    )

    all_indices = np.arange(
        matrix.shape[1],
        dtype=np.int32,
    )

    if (
        feature_indices.size
        == matrix.shape[1]
        and np.array_equal(
            feature_indices,
            all_indices,
        )
    ):
        return matrix

    return np.ascontiguousarray(
        matrix[
            :,
            feature_indices,
        ],
        dtype=np.float32,
    )


search_results = []
selected_feature_sets = {}


def evaluate_search_feature_set(
    method_name,
    feature_indices,
):
    feature_indices = np.asarray(
        feature_indices,
        dtype=np.int32,
    )

    if feature_indices.size == 0:
        raise ValueError(
            f"{method_name} selected "
            "zero features."
        )

    print(
        "\nEvaluating",
        method_name,
        f"({len(feature_indices)} features)...",
    )

    X_fit = select_columns(
        X_fs_train,
        feature_indices,
    )

    X_eval = select_columns(
        X_validation,
        feature_indices,
    )

    search_model = RandomForestClassifier(
        n_estimators=SEARCH_RF_TREES,
        criterion="gini",
        max_features="sqrt",
        min_samples_leaf=1,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    start_time = time.time()

    search_model.fit(
        X_fit,
        y_fs_train,
    )

    predictions = search_model.predict(
        X_eval
    )

    elapsed_time = (
        time.time() - start_time
    )

    result = {
        "Method": method_name,
        "Selected Features": (
            len(feature_indices)
        ),
        "Feature Reduction": (
            X_fs_train.shape[1]
            - len(feature_indices)
        ),
        "Feature Reduction (%)": (
            1
            - (
                len(feature_indices)
                / X_fs_train.shape[1]
            )
        ) * 100,
        "Validation Accuracy": (
            accuracy_score(
                y_validation,
                predictions,
            )
        ),
        "Validation Balanced Accuracy": (
            balanced_accuracy_score(
                y_validation,
                predictions,
            )
        ),
        "Validation Macro F1": f1_score(
            y_validation,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "Search Time (s)": elapsed_time,
    }

    search_results.append(result)

    selected_feature_sets[
        method_name
    ] = feature_indices.copy()

    pd.DataFrame(
        search_results
    ).to_csv(
        SEARCH_RESULTS_PATH,
        index=False,
    )

    print(
        f"Accuracy="
        f"{result['Validation Accuracy']:.4f}"
    )

    print(
        f"Balanced Accuracy="
        f"{result['Validation Balanced Accuracy']:.4f}"
    )

    print(
        f"Macro F1="
        f"{result['Validation Macro F1']:.4f}"
    )

    print(
        f"Time={elapsed_time:.2f}s"
    )

    del search_model
    del predictions
    del X_fit
    del X_eval

    gc.collect()

    return result

In [25]:
all_feature_indices = np.arange(
    X_fs_train.shape[1],
    dtype=np.int32,
)

evaluate_search_feature_set(
    method_name="All Features",
    feature_indices=all_feature_indices,
)

for requested_k in ANOVA_K_VALUES:
    effective_k = min(
        requested_k,
        len(anova_ranked_indices),
    )

    selected_indices = (
        anova_ranked_indices[
            :effective_k
        ]
    )

    evaluate_search_feature_set(
        method_name=(
            f"ANOVA K={effective_k}"
        ),
        feature_indices=selected_indices,
    )

search_results_df = pd.DataFrame(
    search_results
)

search_results_df = (
    search_results_df
    .sort_values(
        [
            "Validation Macro F1",
            "Validation Balanced Accuracy",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(search_results_df)


Evaluating All Features (6373 features)...
Accuracy=0.3053
Balanced Accuracy=0.3025
Macro F1=0.3040
Time=21.74s

Evaluating ANOVA K=250 (250 features)...
Accuracy=0.3342
Balanced Accuracy=0.3369
Macro F1=0.3349
Time=4.51s

Evaluating ANOVA K=500 (500 features)...
Accuracy=0.3466
Balanced Accuracy=0.3481
Macro F1=0.3467
Time=6.84s

Evaluating ANOVA K=750 (750 features)...
Accuracy=0.3438
Balanced Accuracy=0.3442
Macro F1=0.3432
Time=8.81s

Evaluating ANOVA K=1000 (1000 features)...
Accuracy=0.3405
Balanced Accuracy=0.3416
Macro F1=0.3406
Time=10.10s

Evaluating ANOVA K=1250 (1250 features)...
Accuracy=0.3540
Balanced Accuracy=0.3542
Macro F1=0.3532
Time=12.41s

Evaluating ANOVA K=1500 (1500 features)...
Accuracy=0.3398
Balanced Accuracy=0.3401
Macro F1=0.3405
Time=12.23s


,Method,Selected Features,Feature Reduction,Feature Reduction (%),Validation Accuracy,Validation Balanced Accuracy,Validation Macro F1,Search Time (s)
0,ANOVA K=1250,1250,5123,80.386003,0.354027,0.354200,0.353221,12.409038
1,ANOVA K=500,500,5873,92.154401,0.346641,0.348123,0.346719,6.843224
2,ANOVA K=750,750,5623,88.231602,0.343827,0.344240,0.343229,8.808094
3,ANOVA K=1000,1000,5373,84.308803,0.340485,0.341569,0.340641,10.096203
4,ANOVA K=1500,1500,4873,76.463204,0.339782,0.340091,0.340452,12.232671
5,ANOVA K=250,250,6123,96.077201,0.334154,0.336859,0.334946,4.507832
6,All Features,6373,0,0.000000,0.305311,0.302535,0.304030,21.739559


In [26]:
anova_search_df = search_results_df[
    search_results_df["Method"]
    .str.startswith("ANOVA K=")
].copy()

if anova_search_df.empty:
    raise RuntimeError(
        "No ANOVA results were found."
    )

best_anova_method = (
    anova_search_df
    .sort_values(
        [
            "Validation Macro F1",
            "Validation Balanced Accuracy",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .iloc[0]["Method"]
)

compact_anova_df = anova_search_df[
    anova_search_df[
        "Selected Features"
    ] <= 2500
].copy()

if compact_anova_df.empty:
    best_compact_method = (
        best_anova_method
    )
else:
    best_compact_method = (
        compact_anova_df
        .sort_values(
            [
                "Validation Macro F1",
                "Validation Balanced Accuracy",
            ],
            ascending=[
                False,
                False,
            ],
        )
        .iloc[0]["Method"]
    )

CANDIDATE_METHODS = list(
    dict.fromkeys([
        "All Features",
        best_anova_method,
        best_compact_method,
    ])
)

CANDIDATE_FEATURE_SETS = {
    method_name: (
        selected_feature_sets[
            method_name
        ]
    )
    for method_name
    in CANDIDATE_METHODS
}

print("Feature sets moving forward:")

for method_name, indices in (
    CANDIDATE_FEATURE_SETS.items()
):
    reduction = (
        1
        - (
            len(indices)
            / ORIGINAL_FEATURE_COUNT
        )
    ) * 100

    print(
        f"{method_name}: "
        f"{len(indices)} features, "
        f"{reduction:.2f}% reduction"
    )

Feature sets moving forward:
All Features: 6373 features, 0.00% reduction
ANOVA K=1250: 1250 features, 80.39% reduction


In [27]:
RF_CONFIGS = [
    {
        "name": "RF Baseline",
        "n_estimators": 300,
        "criterion": "gini",
        "max_features": "sqrt",
        "min_samples_leaf": 1,
        "class_weight": "balanced",
        "max_samples": None,
    },
    {
        "name": "RF Regularized",
        "n_estimators": 500,
        "criterion": "gini",
        "max_features": "sqrt",
        "min_samples_leaf": 2,
        "class_weight": (
            "balanced_subsample"
        ),
        "max_samples": None,
    },
    {
        "name": "RF Wider Splits",
        "n_estimators": 400,
        "criterion": "gini",
        "max_features": 0.03,
        "min_samples_leaf": 2,
        "class_weight": (
            "balanced_subsample"
        ),
        "max_samples": 0.9,
    },
]

for config in RF_CONFIGS:
    print(config)

{'name': 'RF Baseline', 'n_estimators': 300, 'criterion': 'gini', 'max_features': 'sqrt', 'min_samples_leaf': 1, 'class_weight': 'balanced', 'max_samples': None}
{'name': 'RF Regularized', 'n_estimators': 500, 'criterion': 'gini', 'max_features': 'sqrt', 'min_samples_leaf': 2, 'class_weight': 'balanced_subsample', 'max_samples': None}
{'name': 'RF Wider Splits', 'n_estimators': 400, 'criterion': 'gini', 'max_features': 0.03, 'min_samples_leaf': 2, 'class_weight': 'balanced_subsample', 'max_samples': 0.9}


In [28]:
tuning_results = []

for feature_method, feature_indices in (
    CANDIDATE_FEATURE_SETS.items()
):
    print("\n" + "=" * 70)

    print(
        feature_method,
        "->",
        len(feature_indices),
        "features",
    )

    X_fit = select_columns(
        X_fs_train,
        feature_indices,
    )

    X_eval = select_columns(
        X_validation,
        feature_indices,
    )

    for config in RF_CONFIGS:
        config_name = config["name"]

        parameters = {
            key: value
            for key, value
            in config.items()
            if key != "name"
        }

        print(
            f"\nTraining {config_name}..."
        )

        model = RandomForestClassifier(
            **parameters,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

        start_time = time.time()

        model.fit(
            X_fit,
            y_fs_train,
        )

        predictions = model.predict(
            X_eval
        )

        elapsed_time = (
            time.time() - start_time
        )

        result = {
            "Feature Method": (
                feature_method
            ),
            "Selected Features": (
                len(feature_indices)
            ),
            "Feature Reduction (%)": (
                1
                - (
                    len(feature_indices)
                    / ORIGINAL_FEATURE_COUNT
                )
            ) * 100,
            "Model": config_name,
            "Parameters": parameters,
            "Validation Accuracy": (
                accuracy_score(
                    y_validation,
                    predictions,
                )
            ),
            "Validation Balanced Accuracy": (
                balanced_accuracy_score(
                    y_validation,
                    predictions,
                )
            ),
            "Validation Macro F1": (
                f1_score(
                    y_validation,
                    predictions,
                    average="macro",
                    zero_division=0,
                )
            ),
            "Train/Eval Time (s)": (
                elapsed_time
            ),
        }

        tuning_results.append(result)

        checkpoint_df = pd.DataFrame(
            tuning_results
        )

        checkpoint_to_save = (
            checkpoint_df.copy()
        )

        checkpoint_to_save[
            "Parameters"
        ] = checkpoint_to_save[
            "Parameters"
        ].map(json.dumps)

        checkpoint_to_save.to_csv(
            TUNING_RESULTS_PATH,
            index=False,
        )

        print(
            f"Validation Accuracy: "
            f"{result['Validation Accuracy']:.4f}"
        )

        print(
            f"Validation Balanced Accuracy: "
            f"{result['Validation Balanced Accuracy']:.4f}"
        )

        print(
            f"Validation Macro F1: "
            f"{result['Validation Macro F1']:.4f}"
        )

        print(
            f"Time: {elapsed_time:.2f}s"
        )

        del model
        del predictions
        gc.collect()

    del X_fit
    del X_eval
    gc.collect()


All Features -> 6373 features

Training RF Baseline...
Validation Accuracy: 0.3218
Validation Balanced Accuracy: 0.3332
Validation Macro F1: 0.3208
Time: 52.09s

Training RF Regularized...
Validation Accuracy: 0.3361
Validation Balanced Accuracy: 0.3348
Validation Macro F1: 0.3298
Time: 115.00s

Training RF Wider Splits...
Validation Accuracy: 0.3340
Validation Balanced Accuracy: 0.3355
Validation Macro F1: 0.3301
Time: 194.07s

ANOVA K=1250 -> 1250 features

Training RF Baseline...
Validation Accuracy: 0.3618
Validation Balanced Accuracy: 0.3756
Validation Macro F1: 0.3606
Time: 32.78s

Training RF Regularized...
Validation Accuracy: 0.3713
Validation Balanced Accuracy: 0.3754
Validation Macro F1: 0.3689
Time: 57.35s

Training RF Wider Splits...
Validation Accuracy: 0.3563
Validation Balanced Accuracy: 0.3606
Validation Macro F1: 0.3544
Time: 45.98s


In [29]:
tuning_results_df = pd.DataFrame(
    tuning_results
)

tuning_results_df = (
    tuning_results_df
    .sort_values(
        [
            "Validation Macro F1",
            "Validation Balanced Accuracy",
            "Validation Accuracy",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(
    tuning_results_df[
        [
            "Feature Method",
            "Selected Features",
            "Feature Reduction (%)",
            "Model",
            "Validation Accuracy",
            "Validation Balanced Accuracy",
            "Validation Macro F1",
            "Train/Eval Time (s)",
        ]
    ]
)

BEST_CONFIGURATION = (
    tuning_results_df
    .iloc[0]
    .to_dict()
)

BEST_FEATURE_METHOD = str(
    BEST_CONFIGURATION[
        "Feature Method"
    ]
)

BEST_MODEL_NAME = str(
    BEST_CONFIGURATION["Model"]
)

BEST_PARAMETERS = (
    BEST_CONFIGURATION["Parameters"]
)

print("Winning validation configuration:")

print(
    "Feature method:",
    BEST_FEATURE_METHOD,
)

print(
    "Selected features:",
    BEST_CONFIGURATION[
        "Selected Features"
    ],
)

print(
    "Feature reduction:",
    f"{BEST_CONFIGURATION['Feature Reduction (%)']:.2f}%",
)

print(
    "Model:",
    BEST_MODEL_NAME,
)

print(
    "Parameters:",
    BEST_PARAMETERS,
)

print(
    "Validation Macro F1:",
    f"{BEST_CONFIGURATION['Validation Macro F1']:.4f}",
)

,Feature Method,Selected Features,Feature Reduction (%),Model,Validation Accuracy,Validation Balanced Accuracy,Validation Macro F1,Train/Eval Time (s)
0,ANOVA K=1250,1250,80.386003,RF Regularized,0.371263,0.375425,0.368866,57.349886
1,ANOVA K=1250,1250,80.386003,RF Baseline,0.361766,0.375558,0.360611,32.779541
2,ANOVA K=1250,1250,80.386003,RF Wider Splits,0.356314,0.360589,0.354383,45.977549
3,All Features,6373,0.000000,RF Wider Splits,0.333978,0.335458,0.330059,194.071251
4,All Features,6373,0.000000,RF Regularized,0.336089,0.334839,0.329840,115.001659
5,All Features,6373,0.000000,RF Baseline,0.321843,0.333246,0.320790,52.089362


Winning validation configuration:
Feature method: ANOVA K=1250
Selected features: 1250
Feature reduction: 80.39%
Model: RF Regularized
Parameters: {'n_estimators': 500, 'criterion': 'gini', 'max_features': 'sqrt', 'min_samples_leaf': 2, 'class_weight': 'balanced_subsample', 'max_samples': None}
Validation Macro F1: 0.3689


In [30]:
def refit_final_feature_indices(
    method_name,
    X_training,
    y_training,
):
    total_features = (
        X_training.shape[1]
    )

    if method_name == "All Features":
        return np.arange(
            total_features,
            dtype=np.int32,
        )

    if not method_name.startswith(
        "ANOVA K="
    ):
        raise ValueError(
            f"Unknown feature method: "
            f"{method_name}"
        )

    requested_k = int(
        method_name.split("=")[1]
    )

    training_variances = np.var(
        X_training,
        axis=0,
    )

    training_nonconstant_indices = (
        np.flatnonzero(
            training_variances > 0
        ).astype(np.int32)
    )

    effective_k = min(
        requested_k,
        len(training_nonconstant_indices),
    )

    if (
        len(training_nonconstant_indices)
        == total_features
    ):
        X_anova = X_training
    else:
        X_anova = X_training[
            :,
            training_nonconstant_indices,
        ]

    scores, _ = f_classif(
        X_anova,
        y_training,
    )

    scores = np.nan_to_num(
        scores,
        nan=-np.inf,
        posinf=np.finfo(
            np.float32
        ).max,
        neginf=-np.inf,
    )

    local_ranking = np.argsort(
        scores
    )[::-1]

    selected_local_indices = (
        local_ranking[:effective_k]
    )

    selected_original_indices = (
        training_nonconstant_indices[
            selected_local_indices
        ]
    )

    if X_anova is not X_training:
        del X_anova

    gc.collect()

    return np.asarray(
        selected_original_indices,
        dtype=np.int32,
    )


final_feature_indices = (
    refit_final_feature_indices(
        method_name=BEST_FEATURE_METHOD,
        X_training=X_outer_train,
        y_training=y_outer_train,
    )
)

final_feature_reduction = (
    ORIGINAL_FEATURE_COUNT
    - len(final_feature_indices)
)

final_feature_reduction_percent = (
    final_feature_reduction
    / ORIGINAL_FEATURE_COUNT
) * 100

print(
    "Final feature method:",
    BEST_FEATURE_METHOD,
)

print(
    "Final selected features:",
    len(final_feature_indices),
)

print(
    "Features removed:",
    final_feature_reduction,
)

print(
    "Feature reduction:",
    f"{final_feature_reduction_percent:.2f}%",
)

Final feature method: ANOVA K=1250
Final selected features: 1250
Features removed: 5123
Feature reduction: 80.39%


In [31]:
del X_fs_train
del X_validation
del y_fs_train
del y_validation
del fs_train_groups
del validation_groups
del feature_variances
del anova_scores
del anova_p_values
del anova_ranked_indices

gc.collect()

print("Inner-split arrays released.")

Inner-split arrays released.


In [32]:
X_final_train = select_columns(
    X_outer_train,
    final_feature_indices,
)

X_final_test = select_columns(
    X_test,
    final_feature_indices,
)

final_model = RandomForestClassifier(
    **BEST_PARAMETERS,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

print(
    "Training final model on",
    X_final_train.shape,
)

final_start = time.time()

final_model.fit(
    X_final_train,
    y_outer_train,
)

test_predictions = final_model.predict(
    X_final_test
)

final_elapsed_time = (
    time.time() - final_start
)

test_accuracy = accuracy_score(
    y_test,
    test_predictions,
)

test_balanced_accuracy = (
    balanced_accuracy_score(
        y_test,
        test_predictions,
    )
)

test_macro_f1 = f1_score(
    y_test,
    test_predictions,
    average="macro",
    zero_division=0,
)

final_summary_df = pd.DataFrame([
    {
        "Embedding": DATASET_NAME,
        "Feature Method": (
            BEST_FEATURE_METHOD
        ),
        "Selected Features": len(
            final_feature_indices
        ),
        "Features Removed": (
            final_feature_reduction
        ),
        "Feature Reduction (%)": (
            final_feature_reduction_percent
        ),
        "Model": BEST_MODEL_NAME,
        "Test Accuracy": test_accuracy,
        "Accuracy Change": (
            test_accuracy
            - BASELINE_ACCURACY
        ),
        "Test Balanced Accuracy": (
            test_balanced_accuracy
        ),
        "Balanced Accuracy Change": (
            test_balanced_accuracy
            - BASELINE_BALANCED_ACCURACY
        ),
        "Test Macro F1": (
            test_macro_f1
        ),
        "Macro F1 Change": (
            test_macro_f1
            - BASELINE_MACRO_F1
        ),
        "Final Train/Test Time (s)": (
            final_elapsed_time
        ),
    }
])

display(final_summary_df)

print("\nClassification report:\n")

print(
    classification_report(
        y_test,
        test_predictions,
        labels=TARGET_CLASSES,
        zero_division=0,
    )
)

Training final model on (22735, 1250)


,Embedding,Feature Method,Selected Features,Features Removed,Feature Reduction (%),Model,Test Accuracy,Accuracy Change,Test Balanced Accuracy,Balanced Accuracy Change,Test Macro F1,Macro F1 Change,Final Train/Test Time (s)
0,openSMILE 0.5s,ANOVA K=1250,1250,5123,80.386003,RF Regularized,0.380433,0.051333,0.385397,0.041697,0.3742,0.05,85.393185



Classification report:

                    precision    recall  f1-score   support

           correct       0.34      0.44      0.39      1028
       arched_back       0.33      0.25      0.29       671
      hunched_back       0.37      0.40      0.38       907
          sideways       0.39      0.26      0.31       858
   chest_breathing       0.31      0.21      0.25       852
 over_articulation       0.51      0.47      0.49       692
under_articulation       0.41      0.66      0.51       675

          accuracy                           0.38      5683
         macro avg       0.38      0.39      0.37      5683
      weighted avg       0.38      0.38      0.37      5683



In [33]:
confusion_df = pd.DataFrame(
    confusion_matrix(
        y_test,
        test_predictions,
        labels=TARGET_CLASSES,
    ),
    index=[
        f"True: {label}"
        for label in TARGET_CLASSES
    ],
    columns=[
        f"Predicted: {label}"
        for label in TARGET_CLASSES
    ],
)

display(confusion_df)

,Predicted: correct,Predicted: arched_back,Predicted: hunched_back,Predicted: sideways,Predicted: chest_breathing,Predicted: over_articulation,Predicted: under_articulation
True: correct,457,84,147,62,98,42,138
True: arched_back,170,168,84,52,62,52,83
True: hunched_back,201,62,359,71,61,61,92
True: sideways,174,71,139,227,74,74,99
True: chest_breathing,165,73,133,94,177,75,135
True: over_articulation,93,35,60,45,49,328,82
True: under_articulation,80,10,41,37,47,14,446


In [34]:
search_results_df.to_csv(
    OUTPUT_DIR
    / "anova_search_results.csv",
    index=False,
)

tuning_results_to_save = (
    tuning_results_df.copy()
)

tuning_results_to_save[
    "Parameters"
] = (
    tuning_results_to_save[
        "Parameters"
    ].map(json.dumps)
)

tuning_results_to_save.to_csv(
    OUTPUT_DIR
    / "random_forest_tuning_results.csv",
    index=False,
)

final_summary_df.to_csv(
    OUTPUT_DIR
    / "final_test_result.csv",
    index=False,
)

classification_report_df = pd.DataFrame(
    classification_report(
        y_test,
        test_predictions,
        labels=TARGET_CLASSES,
        output_dict=True,
        zero_division=0,
    )
).transpose()

classification_report_df.to_csv(
    OUTPUT_DIR
    / "classification_report.csv"
)

confusion_df.to_csv(
    OUTPUT_DIR
    / "confusion_matrix.csv"
)

prediction_df = pd.DataFrame({
    "Recording": test_groups,
    "True Label": y_test,
    "Predicted Label": test_predictions,
})

prediction_df.to_csv(
    OUTPUT_DIR
    / "test_predictions.csv",
    index=False,
)

np.savez_compressed(
    OUTPUT_DIR
    / "selected_feature_indices.npz",
    indices=final_feature_indices,
    feature_method=np.asarray(
        BEST_FEATURE_METHOD
    ),
    original_features=np.asarray(
        ORIGINAL_FEATURE_COUNT
    ),
)

joblib.dump(
    {
        "dataset": DATASET_NAME,
        "feature_method": (
            BEST_FEATURE_METHOD
        ),
        "feature_indices": (
            final_feature_indices
        ),
        "model_name": BEST_MODEL_NAME,
        "model_parameters": (
            BEST_PARAMETERS
        ),
        "model": final_model,
        "target_classes": (
            TARGET_CLASSES
        ),
    },
    OUTPUT_DIR
    / "final_model.joblib",
)

with open(
    OUTPUT_DIR
    / "best_configuration.json",
    "w",
) as file:
    json.dump(
        {
            "dataset": DATASET_NAME,
            "feature_method": (
                BEST_FEATURE_METHOD
            ),
            "selected_features": int(
                len(final_feature_indices)
            ),
            "features_removed": int(
                final_feature_reduction
            ),
            "feature_reduction_percent": float(
                final_feature_reduction_percent
            ),
            "model": BEST_MODEL_NAME,
            "parameters": BEST_PARAMETERS,
            "test_accuracy": float(
                test_accuracy
            ),
            "test_balanced_accuracy": float(
                test_balanced_accuracy
            ),
            "test_macro_f1": float(
                test_macro_f1
            ),
        },
        file,
        indent=2,
    )

print(
    "Saved all results to:",
    OUTPUT_DIR.resolve(),
)

Saved all results to: /Users/bhavaykhatri/Desktop/Assignments/audio_data_benchmarking_mml_lab/singBAP_Baselines-and-feature selection/opensmile0.5s/opensmile_0_5s_optimized_feature_selection
